In [ ]:
import pandas as pd
from statsbombpy import sb
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.preprocessing import LabelEncoder
import os
import subprocess
import warnings
import json

warnings.filterwarnings('ignore')

# ==========================================
# 1. SETUP CARTELLE
# ==========================================
RAW_DIR = "../data/raw/"
CLEAN_DIR = "../data/clean/"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)

# Funzione di supporto per eseguire comandi da terminale (es. Git)
def run_cmd(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout + result.stderr

# ==========================================
# 2. INTERFACCIA: MENU PRINCIPALE
# ==========================================
out_main = widgets.Output()

menu_tabs = widgets.ToggleButtons(
    options=['📥 1. Download', '🧹 2. EDA & Pulizia', '🚀 3. Streamlit', '🐙 4. Gestore Git'],
    description='🎯 Azione:',
    button_style='info'
)

# ==========================================
# 3. MODULO 1: DOWNLOAD CENTER
# ==========================================
df_comps = sb.competitions()
comp_names = sorted(df_comps['competition_name'].unique().tolist())

dd_comp = widgets.Dropdown(options=comp_names, description='🏆 Comp:')
dd_season = widgets.Dropdown(options=[], description='📅 Season:')
rb_mode = widgets.RadioButtons(options=['Single Match', 'Full Season Bulk'], value='Single Match', description='⚙️ Mode:')
dd_match = widgets.Dropdown(options=[], description='⚽ Match:', layout=widgets.Layout(width='80%'))
btn_download = widgets.Button(description='🚀 SCARICA DATI', button_style='danger')
out_download = widgets.Output()

def on_comp_change(*args):
    if dd_comp.value:
        seasons = df_comps[df_comps['competition_name'] == dd_comp.value]['season_name'].unique().tolist()
        dd_season.options = sorted(seasons, reverse=True)

def on_season_mode_change(*args):
    if dd_comp.value and dd_season.value:
        row = df_comps[(df_comps['competition_name'] == dd_comp.value) & (df_comps['season_name'] == dd_season.value)].iloc[0]
        if rb_mode.value == 'Single Match':
            dd_match.disabled = False
            try:
                matches = sb.matches(competition_id=row['competition_id'], season_id=row['season_id'])
                dd_match.options = [(f"{r['match_date']} | {r['home_team']} vs {r['away_team']}", r['match_id']) for _, r in matches.iterrows()]
            except:
                dd_match.options = [('Nessun dato', None)]
        else:
            dd_match.disabled = True
            dd_match.options = [('Stagione Completa selezionata', None)]

def esegui_download(b):
    with out_download:
        clear_output()
        print("⏳ Inizio download, attendere...")
        row = df_comps[(df_comps['competition_name'] == dd_comp.value) & (df_comps['season_name'] == dd_season.value)].iloc[0]
        c_name, s_name = dd_comp.value.replace(" ", "_"), dd_season.value.replace("/", "_")
        try:
            if rb_mode.value == 'Single Match' and dd_match.value:
                df = sb.events(match_id=dd_match.value)
                fname = f"statsbomb_{c_name}_{s_name}_match_{dd_match.value}.csv"
            else:
                df_matches = sb.matches(competition_id=row['competition_id'], season_id=row['season_id'])
                df = pd.concat([sb.events(match_id=m['match_id']) for _, m in df_matches.iterrows()], ignore_index=True)
                fname = f"statsbomb_{c_name}_{s_name}_FULL.csv"
            
            path = os.path.join(RAW_DIR, fname)
            df.to_csv(path, index=False)
            print(f"✅ Download completato! Salvato in: {path}")
            aggiorna_lista_raw()
        except Exception as e:
            print(f"❌ Errore: {e}")

dd_comp.observe(on_comp_change, names='value')
dd_season.observe(on_season_mode_change, names='value')
rb_mode.observe(on_season_mode_change, names='value')
btn_download.on_click(esegui_download)

ui_download = widgets.VBox([
    widgets.HTML("<h3>📥 Modulo 1: Download Center</h3>"),
    dd_comp, dd_season, rb_mode, dd_match, btn_download, out_download
])

import json

# ==========================================
# 4. MODULO 2: GENERATORE AMBIENTE EDA
# ==========================================
dd_raw_files = widgets.Dropdown(description='📂 Scegli File:', layout=widgets.Layout(width='60%'))
btn_clean = widgets.Button(description='🏗️ CREA AMBIENTE EDA', button_style='success', icon='flask')
out_clean = widgets.Output()

def aggiorna_lista_raw():
    files = [f for f in os.listdir(RAW_DIR) if f.endswith('.csv')]
    dd_raw_files.options = files if files else ['Nessun file trovato in data/raw/']

def genera_notebook_eda(b):
    with out_clean:
        clear_output()
        if not dd_raw_files.value or dd_raw_files.value.startswith('Nessun'):
            print("❌ Seleziona un file valido dal menu.")
            return
            
        file_selezionato = dd_raw_files.value
        nome_tabella_sql = file_selezionato.replace('.csv', '_clean').lower()
        nome_nuovo_notebook = f"EDA_{file_selezionato.replace('.csv', '')}.ipynb"
        
        print(f"⚙️ Generazione del Notebook di analisi per {file_selezionato} in corso...")
        
        # STRUTTURA DEL NOTEBOOK GENERATO (Celle JSON)
        # Qui iniettiamo il codice delle tue vecchie analisi (Correlazioni, Outlier, LabelEncoder, StandardScaler)
        notebook_content = {
            "nbformat": 4, "nbformat_minor": 5, "metadata": {},
            "cells": [
                {
                    "cell_type": "markdown",
                    "source": [f"# 🔬 Laboratorio EDA: {file_selezionato}\nEsegui le celle per esplorare i dati. Modifica i parametri a tuo piacimento."]
                },
                {
                    "cell_type": "code",
                    "execution_count": None,
                    "source": [
                        "import pandas as pd\n",
                        "import numpy as np\n",
                        "import seaborn as sns\n",
                        "import matplotlib.pyplot as plt\n",
                        "from sklearn.preprocessing import LabelEncoder, StandardScaler\n",
                        "import sys\n",
                        "sys.path.append('../') # Permette di leggere la cartella scripts\n",
                        "from scripts.db_uploader import upload_to_postgres\n\n",
                        f"df = pd.read_csv('../data/raw/{file_selezionato}', low_memory=False)\n",
                        "print(f'Dataset caricato: {df.shape[0]} righe e {df.shape[1]} colonne')\n",
                        "df.head(3)"
                    ],
                    "outputs": []
                },
                {
                    "cell_type": "code",
                    "execution_count": None,
                    "source": [
                        "# 1. ESPLORAZIONE BASE\n",
                        "df.info()\n",
                        "print(df.isnull().sum())"
                    ],
                    "outputs": []
                },
                {
                    "cell_type": "code",
                    "execution_count": None,
                    "source": [
                        "# 2. CODIFICA VARIABILI CATEGORICHE (Label Encoding)\n",
                        "cat_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()\n",
                        "for col in cat_cols:\n",
                        "    df[col] = df[col].fillna('Unknown')\n",
                        "    df[col+'_encoded'] = LabelEncoder().fit_transform(df[col].astype(str))\n",
                        "print('Codifica completata.')"
                    ],
                    "outputs": []
                },
                {
                    "cell_type": "code",
                    "execution_count": None,
                    "source": [
                        "# 3. MATRICE DI CORRELAZIONE\n",
                        "numerical_data = df.select_dtypes(include=np.number)\n",
                        "plt.figure(figsize=(10, 8))\n",
                        "sns.heatmap(numerical_data.corr(), annot=False, cmap='coolwarm')\n",
                        "plt.title('Matrice di Correlazione')\n",
                        "plt.show()"
                    ],
                    "outputs": []
                },
                {
                    "cell_type": "markdown",
                    "source": ["## 🚀 FINE EDA: CARICAMENTO SU DATABASE\nQuando hai finito la pulizia, esegui la cella qui sotto per inviare il Dataset Pulito a PostgreSQL (DBeaver)."]
                },
                {
                    "cell_type": "code",
                    "execution_count": None,
                    "source": [
                        "# Assicurati di aver impostato la password corretta in scripts/db_uploader.py\n",
                        f"upload_to_postgres(df, table_name='{nome_tabella_sql}')"
                    ],
                    "outputs": []
                }
            ]
        }
        
        # Creiamo fisicamente il file del notebook
        percorso_notebook = os.path.join(os.getcwd(), nome_nuovo_notebook)
        with open(percorso_notebook, 'w', encoding='utf-8') as f:
            json.dump(notebook_content, f, indent=4)
            
        print(f"✅ AMBIENTE CREATO CON SUCCESSO!")
        print(f"👉 Apri il file '{nome_nuovo_notebook}' dalla barra laterale sinistra per iniziare lo studio.")

btn_clean.on_click(genera_notebook_eda)

ui_clean = widgets.VBox([
    widgets.HTML("<h3>🧹 Modulo 2: Laboratorio EDA (Manuale)</h3>"),
    widgets.HTML("<i>Scegli il file grezzo. Verrà generato un Notebook su misura per te contenente tutti i tool statistici necessari.</i>"),
    dd_raw_files, btn_clean, out_clean
])

# ==========================================
# 5. MODULO 3: STREAMLIT INFO
# ==========================================
ui_streamlit = widgets.VBox([
    widgets.HTML("<h3>🚀 Modulo 3: Dashboard Streamlit</h3>"),
    widgets.HTML("<p>I dati sono puliti e pronti. Per visualizzare la dashboard, apri il terminale integrato di VS Code ed esegui:</p>"),
    widgets.HTML("<code>streamlit run app.py</code>")
])

# ==========================================
# 6. MODULO 4: GESTORE GIT
# ==========================================
txt_commit = widgets.Text(description='📝 Messaggio:', placeholder='Inserisci il messaggio di commit...')
btn_status = widgets.Button(description='🔍 Status', button_style='info')
btn_pull = widgets.Button(description='⬇️ Pull', button_style='warning')
btn_push = widgets.Button(description='⬆️ Add + Commit + Push', button_style='success')
out_git = widgets.Output()

def git_status(b):
    with out_git:
        clear_output()
        print("🔍 Controllo stato del repository...")
        print(run_cmd("git status"))

def git_pull(b):
    with out_git:
        clear_output()
        print("⬇️ Scaricamento aggiornamenti da GitHub...")
        print(run_cmd("git pull"))

def git_push(b):
    with out_git:
        clear_output()
        messaggio = txt_commit.value if txt_commit.value else "Aggiornamento automatico da Orchestratore"
        print("⬆️ Caricamento su GitHub in corso...")
        print("1. Aggiunta file...")
        print(run_cmd("git add ."))
        print(f"2. Commit con messaggio: '{messaggio}'...")
        print(run_cmd(f'git commit -m "{messaggio}"'))
        print("3. Push su GitHub...")
        print(run_cmd("git push"))
        print("✅ Operazione conclusa!")

btn_status.on_click(git_status)
btn_pull.on_click(git_pull)
btn_push.on_click(git_push)

ui_git = widgets.VBox([
    widgets.HTML("<h3>🐙 Modulo 4: Gestore GitHub</h3>"),
    widgets.HBox([btn_status, btn_pull]),
    widgets.HTML("<br>"),
    widgets.HBox([txt_commit, btn_push]),
    out_git
])

# ==========================================
# 7. GESTORE DELLA NAVIGAZIONE
# ==========================================
def gestisci_navigazione(change):
    with out_main:
        clear_output()
        if change['new'].startswith('📥'):
            on_comp_change()
            on_season_mode_change()
            display(ui_download)
        elif change['new'].startswith('🧹'):
            aggiorna_lista_raw()
            display(ui_clean)
        elif change['new'].startswith('🚀'):
            display(ui_streamlit)
        elif change['new'].startswith('🐙'):
            display(ui_git)

menu_tabs.observe(gestisci_navigazione, names='value')

# Inizializzazione avvio
display(menu_tabs, out_main)
gestisci_navigazione({'new': menu_tabs.value})

ToggleButtons(button_style='info', description='🎯 Azione:', options=('📥 1. Download', '🧹 2. EDA & Pulizia', '🚀…

Output()